# LungPath AI — Train an Object-Detection Model

This notebook downloads **Ahills Workspace / cancer-cell-0smsb, version 1** from Roboflow Universe and trains a YOLOv8 nano object detector in Google Colab. It exports `best.pt` for the FastAPI backend.

> Important: the Roboflow page identifies this as **Object Detection**, not Normal-vs-Tumor image classification. This notebook therefore trains a bounding-box detector. It does not prove that the data are lung-tissue images or that the model can diagnose cancer. Review the class names and example annotations before training.

Dataset source: https://universe.roboflow.com/ahills-workspace/cancer-cell-0smsb (CC BY 4.0; provide attribution when redistributing or publishing results).

In [ ]:
!pip -q install roboflow ultralytics pyyaml

## 1. Add your Roboflow API key safely
In Colab, open the **Secrets** panel (key icon), add a secret named `ROBOFLOW_API_KEY`, and enable notebook access. Do not paste the key into a shared notebook or commit it to GitHub.

In [ ]:
from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY:
    raise RuntimeError('Add ROBOFLOW_API_KEY in Colab Secrets, then enable notebook access.')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('ahills-workspace').project('cancer-cell-0smsb')
dataset_version = project.version(1)
dataset = dataset_version.download('yolov8')
print('Downloaded dataset to:', dataset.location)

## 2. Inspect the labels before training
The public project page has inconsistent image counts between its overview and version page, and the class names shown on the overview are unclear. The downloaded `data.yaml` is the source of truth for the training class names. Stop and review the sample annotations if any class is named `undefined`, consists only of symbols, or does not describe the objects you intend to detect. Do not rename labels unless you have checked the annotation files.

In [ ]:
from pathlib import Path
import yaml

DATA_DIR = Path(dataset.location)
DATA_YAML = DATA_DIR / 'data.yaml'
if not DATA_YAML.exists():
    raise FileNotFoundError(f'Could not find {DATA_YAML}. Check the downloaded export format.')

data_config = yaml.safe_load(DATA_YAML.read_text())
class_names = data_config.get('names', [])
print('Dataset YAML:', DATA_YAML)
print('Class names:', class_names)
print('Number of classes:', data_config.get('nc', len(class_names)))
print('Train:', data_config.get('train'))
print('Validation:', data_config.get('val'))
print('Test:', data_config.get('test'))

bad_names = [str(name) for name in (class_names.values() if isinstance(class_names, dict) else class_names)
             if not str(name).strip() or str(name).strip().lower() == 'undefined' or set(str(name).strip()) <= set('=_- ')]
if bad_names:
    print('WARNING: suspicious class name(s):', bad_names)
    print('Review the dataset labels and class mapping before training. Do not continue blindly.')


In [ ]:
# Display several training images with their bounding-box labels.
from IPython.display import display
from PIL import Image
import random

train_images_dir = DATA_DIR / 'train' / 'images'
train_labels_dir = DATA_DIR / 'train' / 'labels'
sample_images = list(train_images_dir.glob('*'))
print('Training images found:', len(sample_images))
for image_path in random.sample(sample_images, k=min(4, len(sample_images))):
    print(image_path.name)
    display(Image.open(image_path).convert('RGB').resize((480, 480)))
    label_path = train_labels_dir / f'{image_path.stem}.txt'
    print('YOLO labels:', label_path.read_text().strip()[:500] if label_path.exists() else 'No label file')


## 3. Train YOLOv8n
This uses the dataset's existing train/validation split. The dataset page reports 448 train, 64 validation, and 39 test images for version 1 (551 total). The page overview separately reports 327 images, so verify the actual downloaded counts above. If multiple patches/images came from the same patient, image-level splitting may still leak patient information; patient-level grouping is preferable when those identifiers are available.

In [ ]:
import torch
from ultralytics import YOLO

device = 0 if torch.cuda.is_available() else 'cpu'
print('Training device:', device)

model = YOLO('yolov8n.pt')
train_results = model.train(
    data=str(DATA_YAML),
    epochs=50,
    imgsz=640,
    batch=16 if torch.cuda.is_available() else 4,
    workers=2,
    patience=12,
    device=device,
    project='/content/lungpath_runs',
    name='cancer_cell_yolov8n',
    exist_ok=True,
    plots=True,
)

best_weights = Path(model.trainer.best)
print('Best weights:', best_weights)
if not best_weights.exists():
    raise FileNotFoundError('Training finished but best.pt was not found.')

## 4. Evaluate on the held-out test split
Record the metrics and inspect the confusion matrix / prediction plots. A good validation score alone does not demonstrate performance on independent patients or clinical usefulness.

In [ ]:
best_model = YOLO(str(best_weights))
test_metrics = best_model.val(data=str(DATA_YAML), split='test', imgsz=640, device=device, plots=True)
print('mAP50:', float(test_metrics.box.map50))
print('mAP50-95:', float(test_metrics.box.map))
print('Precision:', float(test_metrics.box.mp))
print('Recall:', float(test_metrics.box.mr))

## 5. Export weights for the backend
Download `best.pt`. Put it in the project folder `LungPath_AI/model/best.pt`. The FastAPI backend expects a YOLO weights file at that location by default (or use the `MODEL_PATH` environment variable).

In [ ]:
from google.colab import files
files.download(str(best_weights))

## Attribution and interpretation
Dataset: **cancer cell** by **Ahills Workspace**, Roboflow Universe, version 1, CC BY 4.0. Source: https://universe.roboflow.com/ahills-workspace/cancer-cell-0smsb. Include this attribution when sharing the dataset or derived model.

This is an experimental object detector trained on the downloaded annotations. It is not a validated lung-cancer detector, does not classify whole slides as Tumor vs Normal, and must not be used for diagnosis or patient-care decisions.